# Detecting Unlearning Traces on TOFU

Same idea as `reproduction.ipynb` (train a probe on model outputs to tell an **unlearned** model from its **original**), but on the **TOFU** fictitious-author unlearning benchmark instead of WMDP.

**Setup (per plan):**
- **Original** = Phi-1.5 fine-tuned on all TOFU (`checkpoint-625`, before any unlearning).
- **Unlearned** = same repo after `grad_ascent` on the `forget10` split (latest checkpoint branch).
- **Prompts** = questions from the TOFU `forget10` split (the info the unlearned model is supposed to forget).
- **Feature** = pre-logit (lm_head input) activations, mean-pooled over the generated continuation.
- **Probe** = small MLP, binary: label 0 = original, 1 = unlearned.
- Optional control: repeat on `world_facts` (unrelated) — the trace should be strong on forget, weak on world facts.

Runs on a **Colab T4** 

## 1. Install + imports

In [ ]:
# !pip install -U transformers datasets scikit-learn accelerate bitsandbytes huggingface_hub

In [1]:
import importlib
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HAS_GPU = torch.cuda.is_available()
DEFAULT_DTYPE = torch.bfloat16 if (HAS_GPU and torch.cuda.is_bf16_supported()) else torch.float16
if HAS_GPU:
    print(f"[gpu] {torch.cuda.get_device_name(0)} | "
          f"VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("[warn] no CUDA GPU — this needs a Colab GPU runtime.")

[gpu] Tesla T4 | VRAM 15.6 GB


## 2. bitsandbytes-safe model loading

Reused from `reproduction.py`. Older `accelerate` calls `model.to(device)` inside `dispatch_model` for a single-GPU map, which 4-bit models reject. We patch `dispatch_model` (in `transformers.modeling_utils`, where it's actually called) to move only the **non-bnb** params/buffers to the GPU and leave the quantized weights in place. Harmless when not using 4-bit.

In [ ]:
def _quant_config(load_4bit):
    if not load_4bit:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DEFAULT_DTYPE, bnb_4bit_use_double_quant=True,
    )


def _model_load_kwargs(load_4bit):
    if load_4bit:
        return dict(quantization_config=_quant_config(True),
                    device_map="auto", low_cpu_mem_usage=True)
    return dict(torch_dtype=DEFAULT_DTYPE, device_map="auto")


def _model_device(model):
    return next(model.parameters()).device


def _patch_accelerate_for_bnb():
    def _is_quantized(model):
        cfg = getattr(model, "config", None)
        return bool(
            getattr(model, "is_quantized", False)
            or getattr(model, "is_loaded_in_4bit", False)
            or getattr(model, "is_loaded_in_8bit", False)
            or getattr(model, "hf_quantizer", None) is not None
            or getattr(cfg, "quantization_config", None) is not None
        )

    def _target_device(device_map):
        if isinstance(device_map, dict):
            for d in device_map.values():
                if d not in ("cpu", "disk", None):
                    return torch.device(f"cuda:{d}" if isinstance(d, int) else d)
        return torch.device("cuda:0")

    def _place_non_bnb(model, target):
        bnb_types = ("Params4bit", "Int8Params")
        for module in model.modules():
            for name, p in list(module._parameters.items()):
                if p is None or p.__class__.__name__ in bnb_types:
                    continue
                if p.device.type != "cuda":
                    module._parameters[name] = torch.nn.Parameter(
                        p.data.to(target), requires_grad=p.requires_grad)
            for name, b in list(module._buffers.items()):
                if b is not None and b.device.type != "cuda":
                    module._buffers[name] = b.to(target)

    def _make_safe(original):
        def _safe_dispatch(model, device_map=None, **kwargs):
            if _is_quantized(model):
                _place_non_bnb(model, _target_device(device_map))
                if device_map is not None:
                    model.hf_device_map = device_map
                return model
            return original(model, device_map=device_map, **kwargs)
        return _safe_dispatch

    for mod_name in ("transformers.modeling_utils", "accelerate.big_modeling", "accelerate"):
        try:
            mod = importlib.import_module(mod_name)
        except ImportError:
            continue
        fn = getattr(mod, "dispatch_model", None)
        if fn is None or getattr(fn, "_bnb_patched", False):
            continue
        safe = _make_safe(fn)
        safe._bnb_patched = True
        mod.dispatch_model = safe


_patch_accelerate_for_bnb()
print("bnb dispatch patch installed.")

bnb dispatch patch installed.


## 3. Resolve the TOFU checkpoints

The Phi TOFU repo stores each unlearning checkpoint as a **git branch** (`checkpoint-X`). `checkpoint-625` is the TOFU-finetuned model **before** unlearning (our *original*); the **latest** `checkpoint-X` branch is the most-unlearned model (our *unlearned*). We discover the branches automatically so we don't hardcode the wrong number.

In [ ]:
TOFU_REPO = "locuslab/phi_grad_ascent_1e-05_forget10"  # grad_ascent, forget10
ORIGINAL_REVISION = "checkpoint-625"                    # TOFU-finetuned, pre-unlearn
UNLEARNED_REVISION = None  # None -> auto-pick the latest checkpoint branch


def list_checkpoint_branches(repo):
    from huggingface_hub import list_repo_refs
    refs = list_repo_refs(repo)
    names = [b.name for b in refs.branches if b.name.startswith("checkpoint-")]
    # sort by the numeric suffix
    names.sort(key=lambda n: int(n.split("-")[-1]))
    return names


branches = list_checkpoint_branches(TOFU_REPO)
print("checkpoint branches:", branches)

if UNLEARNED_REVISION is None:
    candidates = [b for b in branches if b != ORIGINAL_REVISION]
    UNLEARNED_REVISION = candidates[-1] if candidates else branches[-1]

if ORIGINAL_REVISION not in branches:
    # fall back to the earliest branch as the 'original'
    ORIGINAL_REVISION = branches[0]

print(f"original  -> {TOFU_REPO} @ {ORIGINAL_REVISION}")
print(f"unlearned -> {TOFU_REPO} @ {UNLEARNED_REVISION}")

checkpoint branches: ['checkpoint-12', 'checkpoint-24', 'checkpoint-36', 'checkpoint-48', 'checkpoint-60']
original  -> locuslab/phi_grad_ascent_1e-05_forget10 @ checkpoint-12
unlearned -> locuslab/phi_grad_ascent_1e-05_forget10 @ checkpoint-60


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## 4. Build TOFU prompts

Load the TOFU `forget10` split and use its `question` field, formatted the way TOFU models were trained (`Question: ... \nAnswer:`). We also load `world_facts` for the optional control.

In [4]:
def build_tofu_prompts(split, num_samples, seed=42):
    from datasets import load_dataset
    ds = load_dataset("locuslab/TOFU", split)["train"]
    import random
    rng = random.Random(seed)
    idx = rng.sample(range(len(ds)), min(num_samples, len(ds)))
    return [f"Question: {ds[i]['question']}\nAnswer:" for i in idx]


NUM_SAMPLES = 100
forget_prompts = build_tofu_prompts("forget10", NUM_SAMPLES)
print(f"built {len(forget_prompts)} forget10 prompts")
print("example:\n", forget_prompts[0])

built 100 forget10 prompts
example:
 Question: Does Aysha Al-Hashim have any book series in her portfolio?
Answer:


## 5. Pre-logit activation feature + MLP probe

Identical to the WMDP notebook: hook the input of `lm_head` during greedy generation, take the last position at each step, mean-pool over the generated tokens.

In [5]:
@torch.no_grad()
def get_pre_logit_activations(model, tokenizer, prompt, max_new_tokens=32):
    inputs = tokenizer(prompt, return_tensors="pt").to(_model_device(model))
    activations = []

    def hook_fn(module, inp, out):
        activations.append(inp[0][:, -1, :].detach().float().cpu())

    handle = model.lm_head.register_forward_hook(hook_fn)
    try:
        model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                       pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    finally:
        handle.remove()
    return torch.cat(activations, dim=0).mean(dim=0)


def build_activation_features(repo, revision, prompts, max_new_tokens=32, load_4bit=False):
    tokenizer = AutoTokenizer.from_pretrained(repo, revision=revision)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        repo, revision=revision, **_model_load_kwargs(load_4bit))
    model.eval()
    feats = [get_pre_logit_activations(model, tokenizer, p, max_new_tokens) for p in prompts]
    del model
    torch.cuda.empty_cache()
    return torch.stack(feats, dim=0).numpy().astype(np.float32)

In [6]:
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def _loader(X, y, bs, shuffle):
    ds = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                       torch.as_tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)


def train_mlp(X, y, epochs=50, lr=1e-3, batch_size=32, weight_decay=1e-4, device=DEVICE):
    X = np.asarray(X, np.float32); y = np.asarray(y, np.float32)
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)
    tr, va, te = (_loader(X_tr, y_tr, batch_size, True), _loader(X_val, y_val, batch_size, False),
                  _loader(X_te, y_te, batch_size, False))
    model = BinaryClassifier(X.shape[1]).to(device)
    crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    def ev(loader):
        model.eval(); preds, gts = [], []
        with torch.no_grad():
            for xb, yb in loader:
                preds.append((torch.sigmoid(model(xb.to(device))) > 0.5).long().cpu())
                gts.append(yb.long())
        return accuracy_score(torch.cat(gts), torch.cat(preds))

    best_val, best_state = -1.0, None
    for _ in range(epochs):
        model.train()
        for xb, yb in tr:
            opt.zero_grad()
            crit(model(xb.to(device)), yb.to(device)).backward()
            opt.step()
        v = ev(va)
        if v > best_val:
            best_val = v
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    test_acc = ev(te)
    print(f"  best val acc: {best_val:.4f} | test acc: {test_acc:.4f}")
    return test_acc

## 6. Run: original vs unlearned on the forget set

Generate activations from both checkpoints on the same forget-set questions, then train the probe. Phi-1.5 is small; set `LOAD_4BIT = True` only if you hit OOM.

In [7]:
LOAD_4BIT = False
ACT_NEW_TOKENS = 32

print("building ORIGINAL activations...")
Xo = build_activation_features(TOFU_REPO, ORIGINAL_REVISION, forget_prompts,
                               max_new_tokens=ACT_NEW_TOKENS, load_4bit=LOAD_4BIT)
print("building UNLEARNED activations...")
Xu = build_activation_features(TOFU_REPO, UNLEARNED_REVISION, forget_prompts,
                               max_new_tokens=ACT_NEW_TOKENS, load_4bit=LOAD_4BIT)

X = np.concatenate([Xo, Xu], axis=0)
y = np.array([0] * len(Xo) + [1] * len(Xu))
print(f"features: {X.shape} | labels: {y.shape}")

print("\n[forget10] original vs unlearned detection:")
forget_acc = train_mlp(X, y)

building ORIGINAL activations...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

...

building UNLEARNED activations...


config.json:   0%|          | 0.00/990 [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

features: (200, 2048) | labels: (200,)

[forget10] original vs unlearned detection:
  best val acc: 1.0000 | test acc: 1.0000


## 7. (Optional) Control: world_facts

TOFU-native sanity check. Unlearning `forget10` targets the fictitious authors, so the trace should be **strong** on forget-set questions and **weaker** on unrelated `world_facts` (both models should behave the same there). If forget accuracy ≫ world-facts accuracy, the probe is picking up the *unlearning*, not generic checkpoint noise.

In [8]:
RUN_CONTROL = True

if RUN_CONTROL:
    wf_prompts = build_tofu_prompts("world_facts", NUM_SAMPLES)
    print(f"built {len(wf_prompts)} world_facts prompts\n")
    Wo = build_activation_features(TOFU_REPO, ORIGINAL_REVISION, wf_prompts,
                                   max_new_tokens=ACT_NEW_TOKENS, load_4bit=LOAD_4BIT)
    Wu = build_activation_features(TOFU_REPO, UNLEARNED_REVISION, wf_prompts,
                                   max_new_tokens=ACT_NEW_TOKENS, load_4bit=LOAD_4BIT)
    Xw = np.concatenate([Wo, Wu], axis=0)
    yw = np.array([0] * len(Wo) + [1] * len(Wu))
    print("[world_facts] original vs unlearned detection (control):")
    world_acc = train_mlp(Xw, yw)
    print(f"\nSUMMARY: forget10 acc = {forget_acc:.3f} | world_facts acc = {world_acc:.3f}")
    print("Higher forget10 vs world_facts => trace is specific to the unlearned content.")

world_facts.json:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117 [00:00<?, ? examples/s]

built 100 world_facts prompts



Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

[world_facts] original vs unlearned detection (control):
  best val acc: 1.0000 | test acc: 1.0000

SUMMARY: forget10 acc = 1.000 | world_facts acc = 1.000
Higher forget10 vs world_facts => trace is specific to the unlearned content.


## 8. Notes / how to extend

- **Different unlearning method:** change `TOFU_REPO` to another checkpoint from the [TOFU Unlearned Models collection](https://huggingface.co/collections/locuslab/tofu-unlearned-models) (e.g. `grad_diff`, `KL`, `idk`/preference-optimization). The branch/revision machinery is the same.
- **Different forget split:** set the split in `build_tofu_prompts` to `forget01` / `forget05` (fewer samples; probe may be noisier).
- **Llama2-7B TOFU** (`locuslab/tofu_ft_llama2-7b` + a Llama2 unlearned checkpoint): bigger + gated; use `LOAD_4BIT = True` and accept the Llama2 license on HF first.
- **Text features (LLM2Vec)** instead of activations: too heavy for a T4; use an A100 as in the WMDP notebook.